# Loss Reserving: Chain Ladder and Bornhuetter-Ferguson

**Dataset**: CAS Schedule P — Private Passenger Auto, State Farm (1998–2007)  
**Methods**: Chain Ladder, Bornhuetter-Ferguson, Credibility Theory  
**Key question**: How much should we reserve for claims already incurred but not yet fully paid?


## 1. The Loss Reserving Problem

Insurance claims take time to develop. A policyholder files a claim today, but the insurer may not make the final payment for months or years. At any point in time, the insurer must estimate:

- **Case reserves**: reserves set for known, reported claims
- **IBNR** (Incurred But Not Reported): reserves for claims that have occurred but haven't been reported yet

The total reserve = Case Reserves + IBNR.

Underreserving → insolvency risk. Overreserving → distorted profitability and tax implications.

### The Loss Triangle

The core data structure is the **loss triangle** — cumulative paid losses by accident year (rows) and development age (columns):

```
Accident Year | 12 months | 24 months | 36 months | ... | 120 months
    1998      |   10,234  |   18,412  |   24,301  | ... |   31,200   ← fully developed
    1999      |   11,801  |   20,144  |   26,891  | ... |      ?     ← need to estimate
    2000      |   12,500  |   21,300  |      ?    | ... |      ?     ← need to estimate
    2003      |   14,322  |      ?    |      ?    | ... |      ?     ← most immature
```

The goal: fill in the lower-right `?` values to estimate ultimate losses.


## 2. Load Data and Build Triangle

In [ ]:
import sys
sys.path.insert(0, 'actuarial')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chain_ladder import load_cas_triangle, truncate_triangle, ChainLadder, list_companies
from bornhuetter_ferguson import BornhuetterFerguson

# Show available companies
companies = list_companies('data/cas_triangles/ppauto_pos98-07.csv')
print('Top 10 companies by earned premium:')
print(companies.head(10).to_string(index=False))

In [ ]:
# Load largest company (State Farm)
triangle, premium, grname = load_cas_triangle('data/cas_triangles/ppauto_pos98-07.csv')

print(f'\nFull triangle shape: {triangle.shape}')
print('\nFull cumulative paid loss triangle ($):')
print(triangle.to_string())

In [ ]:
# Truncate to simulate mid-development valuation at lag 6
# This mirrors a real valuation where recent accident years are immature
triangle_trunc = truncate_triangle(triangle, valuation_lag=6)

print('Truncated triangle (simulating valuation at end of development lag 6):')
print(triangle_trunc.to_string())
print('\nNaN = losses not yet observed at valuation date')

## 3. Chain Ladder Method

The chain ladder method projects losses to ultimate using **age-to-age development factors** (LDFs).

### Step 1: Compute LDFs

The volume-weighted LDF from lag $k$ to lag $k+1$ is:

$$f_k = \frac{\sum_i C_{i,k+1}}{\sum_i C_{i,k}}$$

where $C_{i,k}$ is cumulative paid losses for accident year $i$ at development lag $k$, and the sum runs over all accident years where both $C_{i,k}$ and $C_{i,k+1}$ are observed.

### Step 2: Compute CDFs to Ultimate

The cumulative development factor (CDF) from lag $k$ to ultimate is:

$$F_k = f_k \times f_{k+1} \times \cdots \times f_{K} \times f_{\text{tail}}$$

We assume a tail factor of 1.0 (fully developed at lag 10 for auto liability).

### Step 3: Project to Ultimate

$$\hat{C}_{i,\infty} = C_{i, k_i} \times F_{k_i}$$

where $k_i$ is the latest observed lag for accident year $i$.

### Step 4: IBNR

$$\text{IBNR}_i = \hat{C}_{i,\infty} - C_{i, k_i}$$


In [ ]:
cl = ChainLadder(triangle_trunc, premium)
cl.fit()
cl.development_factors()
print()
cl.summary()

In [ ]:
fig = cl.plot(grname=grname)
plt.show()

### Interpreting the Development Factors

- **LDF 1→2 ≈ 1.66**: losses nearly double from year 1 to year 2 — most IBNR emerges early
- **LDF 5→6 ≈ 1.02**: almost all development has occurred by lag 5; only 2% remains
- **CDF at lag 1 ≈ 2.24**: the most immature accident year will ultimately pay 2.24× what's been paid so far

The **weakness** of chain ladder: for immature accident years, a small error in early reported losses gets multiplied by a large CDF, potentially swinging the reserve estimate significantly. This is where Bornhuetter-Ferguson helps.


## 4. Bornhuetter-Ferguson Method

The BF method addresses chain ladder's instability for immature years by blending observed development with an *a priori* expected loss ratio (ELR).

### The Formula

$$\text{IBNR}_{\text{BF},i} = P_i \times \text{ELR} \times \left(1 - \frac{1}{F_{k_i}}\right)$$

$$\hat{C}_{i,\infty}^{\text{BF}} = C_{i,k_i} + \text{IBNR}_{\text{BF},i}$$

Where:
- $P_i$ = earned premium for accident year $i$
- $\text{ELR}$ = expected loss ratio (from pricing assumptions or industry benchmarks)
- $\left(1 - 1/F_k\right)$ = **unreported factor** — the fraction of ultimate losses not yet paid

### Credibility Interpretation

BF is equivalent to a credibility-weighted blend:

$$\hat{C}_{i,\infty}^{\text{BF}} = z_i \cdot \hat{C}_{i,\infty}^{\text{CL}} + (1 - z_i) \cdot P_i \cdot \text{ELR}$$

where $z_i = 1/F_{k_i}$ is the **credibility weight** = fraction of losses already reported.

- **Mature years** ($z \to 1$): BF converges to chain ladder — data is fully credible
- **Immature years** ($z \to 0$): BF relies on the a priori ELR — observed data has little credibility

This is exactly analogous to Bühlmann credibility in pricing:
$\hat{\mu} = z \bar{X} + (1-z) \mu_0$


In [ ]:
bf = BornhuetterFerguson(cl)
bf.fit()
bf.summary()
print()
bf.credibility_weights()

In [ ]:
fig = bf.plot(grname=grname)
plt.show()

## 5. ELR Sensitivity Analysis

The BF reserve depends on the actuary's choice of ELR. How sensitive is the total IBNR to this assumption?


In [ ]:
elr_values = [0.60, 0.65, 0.70, 0.756, 0.80, 0.85]
rows = []
for elr in elr_values:
    bf_test = BornhuetterFerguson(cl, elr=elr)
    bf_test.fit()
    rows.append({
        'ELR': f'{elr:.1%}',
        'Total IBNR (BF)': bf_test.results['IBNR_BF'].sum(),
        'vs CL': bf_test.results['IBNR_BF'].sum() - cl.results['IBNR'].sum(),
    })

sens_df = pd.DataFrame(rows)
sens_df['Total IBNR (BF)'] = sens_df['Total IBNR (BF)'].apply(lambda x: f'${x:,.0f}')
sens_df['vs CL'] = sens_df['vs CL'].apply(lambda x: f'${x:+,.0f}')
print('ELR Sensitivity — Total IBNR:')
print(sens_df.to_string(index=False))
print(f'\nChain ladder IBNR (reference): ${cl.results["IBNR"].sum():,.0f}')

## 6. Key Takeaways

1. **Chain ladder** is simple and data-driven but amplifies noise for immature accident years (high CDF × small early losses = unstable estimate).

2. **Bornhuetter-Ferguson** stabilizes immature years by substituting the a priori ELR for the unreported portion of losses. For mature years it converges to chain ladder.

3. **Credibility weights** make the BF method interpretable: the 2003 accident year (lag 1) has z = 0.45 — meaning only 45% of its reserve comes from observed data; 55% from the a priori.

4. **ELR sensitivity** matters most for the most immature years. A 10pp change in ELR can shift total IBNR by ~$1M on this book, driven almost entirely by the lag-1 accident year.

5. The methods agree for mature years — divergence signals either model error or true uncertainty about recent loss trends.
